# VeritasAI · 00 · Prepare data (run ONCE)
CPU-only is enough here – set *Accelerator = None* on Kaggle to save GPU quota.
Output: 16 face crops per video as one small JPEG strip + identity-safe train/val/test CSVs.

In [ ]:
# ==== 0. SETUP (same in every notebook) ===================================
# Kaggle: Settings -> Accelerator = GPU T4 x2 (or None for notebook 00), Internet = ON
# Colab : Runtime -> Change runtime type -> T4 GPU
import os, sys
KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if KAGGLE else "/content"
REPO_URL = "https://github.com/Bhoomified/VeritasAI.git"   # your repo (push the new ml/ files first!)
os.chdir(WORK)
if not os.path.exists("VeritasAI"):
    os.system(f"git clone -q {REPO_URL}")
os.chdir(f"{WORK}/VeritasAI"); os.system("git pull -q")
os.system('pip -q install "opencv-python-headless==4.10.0.84" onnx onnxruntime tabulate')
if not KAGGLE:   # Colab: keep outputs on Google Drive so a disconnect never loses work
    from google.colab import drive; drive.mount("/content/drive")
STORE = "/kaggle/working" if KAGGLE else "/content/drive/MyDrive/veritas"
os.makedirs(STORE, exist_ok=True)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
# ==== 1. dataset locations (edit) ========================================
# Upload each dataset to Kaggle as a PRIVATE dataset (respect the licence: no redistribution).
FFPP   = "/kaggle/input/ffpp-c23"          # contains original_sequences/ and manipulated_sequences/
CELEB  = "/kaggle/input/celeb-df-v2"       # contains Celeb-real/ YouTube-real/ Celeb-synthesis/ List_of_testing_videos.txt
DFDC   = "/kaggle/input/deepfake-detection-challenge/train_sample_videos"   # optional, attach the competition data
DATA   = f"{STORE}/data"; FACES = f"{STORE}/faces"

In [ ]:
# ==== 2. manifests: split BY IDENTITY, balance 1 fake : 1 real ===============
!python -m ml.prepare manifest --layout ffpp --root {FFPP} --out {DATA}/ffpp --fake-ratio 1
!python -m ml.prepare manifest --layout celebdf --root {CELEB} --out {DATA}/celebdf --test-only --fake-ratio 0
# optional third test set:
# !python -m ml.prepare manifest --layout dfdc --root {DFDC} --out {DATA}/dfdc --test-only --fake-ratio 1

In [ ]:
# ==== 3. faces: decode each video once, YuNet face crop, 16 frames ==========
!python -m ml.prepare faces --csv {DATA}/ffpp/train.csv {DATA}/ffpp/val.csv {DATA}/ffpp/test.csv --out {FACES}/ffpp --frames 16
!python -m ml.prepare faces --csv {DATA}/celebdf/test.csv --out {FACES}/celebdf --frames 16
!du -sh {FACES}/*

In [ ]:
# ==== 4. eyeball a few strips (ALWAYS check your data) =====================
import glob, cv2, matplotlib.pyplot as plt
for p in sorted(glob.glob(f"{FACES}/ffpp/*.jpg"))[:3]:
    plt.figure(figsize=(16, 1.5)); plt.imshow(cv2.imread(p)[:, :, ::-1][:, :224*8]); plt.axis("off")

**Kaggle:** click *Save Version → Save & Run All*. Then in notebooks 01-04 use *Add Input → Your Work → this notebook* (title this notebook **veritas-faces** so the path is `/kaggle/input/veritas-faces/faces`).
**Colab:** the strips are already on your Google Drive (`MyDrive/veritas/faces`).